# Publish to BigQuery

## Objective

Send a completed run to BigQuery and safely repeat the publication.

Run notebooks 2 through 4 first. This writes saved forecasts and scores, including every retained tuning window, without fitting or rescoring. You need write access to your chosen destination. These adapters publish results; they do not add source-data ingestion from this database.

Every row carries study ID/hash, workflow run ID, stage, origin and horizon. Tuning rows also carry search ID, trial ID, attempt ID and zero-based window index. Final validation/test rows have null tuning identifiers. Join using the complete identity and null-safe comparisons. Recorded scores remain in `evidence_json`.

Install `autoforecast[bigquery]` and configure Google Application Default Credentials. Set `AUTOFORECAST_BIGQUERY_DATASET` to your `project.dataset` and `AUTOFORECAST_BIGQUERY_LOCATION` to its region. Optionally set `AUTOFORECAST_TABLE_PREFIX` to choose distinct example tables. Do not paste credentials into the notebook. Queries and storage may incur charges.


## Step 1: Choose a place for the example files

We will keep data and results outside the library folder. The printed path shows where to find them. Set the environment variable only if you want a different folder.


We also turn on progress messages, so we can see which step is running.


In [ ]:
# Import the tools used in this step.
import os
from pathlib import Path

from autoforecast import configure_logging

# Choose the workspace without changing the library checkout.
work = (
    Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", Path.home() / "autoforecast-tutorials"))
    .expanduser()
    .resolve()
)
# Create the folder if this is your first visit.
work.mkdir(parents=True, exist_ok=True)
print(work)

# Show the current step while the library is working.
configure_logging()


### Understand the output

You should see the full workspace path. Open that folder to find the files created in these lessons. This path is a folder, not a forecast result.


Turning logging on does not run a forecast. Later library calls will show their steps. A stage count tells us how many steps have finished, not how much time is left.


## Step 2: Start the data engine

We will start a small local Spark session with two workers. Spark reads and checks the study data. UTC keeps date and time handling consistent.


In [ ]:
# Import the tools used in this step.
from pyspark.sql import SparkSession

# Use two local workers for this small example.
spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

### Understand the output

Spark may print startup messages. The cell should finish without an exception. No forecasts have been made yet.


## Step 3: Open the registry and saved run

We will load the study registry and completed workflow. The saved forecasts will be copied to BigQuery.


In [ ]:
# Import the tools used in this step.
from autoforecast.study_registration import StudyRegistry
from autoforecast.study_workflow import load_forecast_workflow

# Open the registry described by our saved study settings.
registry = StudyRegistry.from_config(work / "study.yaml", spark=spark)

# Check and open the saved run without fitting again.
result = load_forecast_workflow((work / "latest-run.txt").read_text().strip())

### Understand the output

You may see verification messages while the saved files are checked. We have loaded the same saved run that will be copied to BigQuery.


## Step 4: Connect and publish

We will read the project, dataset and region from environment settings. Then we will create example tables, publish twice, and read back the saved windows. Choose a table prefix reserved for this tutorial.


In [ ]:
# Import the cloud client and result destination.
from autoforecast.integrations.bigquery import BigQueryResults
from google.cloud import bigquery

# Read the existing dataset, its region, and a table prefix from the environment.
dataset = os.environ["AUTOFORECAST_BIGQUERY_DATASET"]
location = os.environ["AUTOFORECAST_BIGQUERY_LOCATION"]
prefix = os.environ.get("AUTOFORECAST_TABLE_PREFIX", "tutorial")
# Use existing cloud credentials and close the client after use.
with bigquery.Client(project=dataset.split(".")[0]) as client:
    destination = BigQueryResults(
        client,
        dataset=dataset,
        location=location,
        results=f"{prefix}_results",
        forecasts=f"{prefix}_forecasts",
    )
    # Create only missing destination tables; keep any earlier results.
    destination.create_tables()
    where = "WHERE study_id = @study_id AND study_sha = @study_sha AND run_id = @run_id"
    for attempt in range(2):
        publication = registry.publish_results(
            result.study_id, result.path, destination=destination
        )
        # Pass identifiers as query parameters, not as pieces of SQL text.
        query_config = bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("study_id", "STRING", result.study_id),
                bigquery.ScalarQueryParameter("study_sha", "STRING", publication["study_sha256"]),
                bigquery.ScalarQueryParameter("run_id", "STRING", result.run_id),
            ]
        )
        # Read only this run and retain each trial/window identity in the preview.
        query = (
            f"SELECT stage, trial_id, window_index, origin, horizon "
            f"FROM `{dataset}.{prefix}_results` "
            + where
            + " ORDER BY stage, trial_id, window_index, origin"
        )
        rows = list(client.query(query, location=location, job_config=query_config).result())
        forecast_count = next(
            iter(
                client.query(
                    f"SELECT COUNT(*) AS count FROM `{dataset}.{prefix}_forecasts` " + where,
                    location=location,
                    job_config=query_config,
                ).result()
            )
        )["count"]
        print(
            "Publication",
            attempt + 1,
            publication["status"],
            "result rows:",
            len(rows),
            "forecast rows:",
            forecast_count,
        )
    # Print the small scoped preview after the second publication.
    for row in rows:
        print(dict(row))

### Understand the output

Each publication should report `published`, 6 result rows and 84 forecast rows for this run. The second call should show the same counts, even if older runs are in the tables. The preview contains four tuning fits, validation and test. `trial_id` distinguishes the two tuning settings, and `window_index` is 0 or 1 for their earlier windows. These fields are empty (`null` or `None`) for validation and test. `origin` is the last training date; `horizon: 7` means seven future days. These six result rows summarize fits; the 84 forecast rows hold individual product/date predictions.


At INFO level, stage messages and a progress panel also show the work as it happens. The publication status and row counts described above is the result to keep. A completed-stage percentage counts steps; it does not predict how much time remains.


Each destination write is atomic, but the two tables are not one transaction. Forecasts publish first; result rows mark a completed publication. Stable job IDs and row comparisons prevent duplicate retries. Changed rows under an existing identity are refused.

Staging tables are deleted after use and expire after one day if the process is interrupted. The two destination tables remain until you delete them. Deleting tables does not erase service-managed job or audit records. The adapter needs permissions to create/delete staging tables, load data, query, and write destination tables.

You also need permission to run BigQuery jobs in the destination project (`bigquery.jobs.create`). Publication writes use that project so retries share the same job identity, even when clients have different default projects.


## Step 5: Close this Spark session

We are finished with Spark in this notebook. Closing it frees resources before the next lesson.


In [ ]:
# Release Spark resources before moving to another notebook.
spark.stop()

### Understand the output

No output is expected. Spark is now closed. Start a fresh kernel when the next notebook asks you to.
